In [0]:
%pip install boto3 requests

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
access_key = dbutils.secrets.get(scope = "rearc_s3", key = "aws_access_key")
secret_key = dbutils.secrets.get(scope = "rearc_s3", key = "aws_secret_key")
REGION = "us-east-1"
BUCKET = 'foruhar-rearc-quest'
BASE_URL = "https://download.bls.gov/pub/time.series/pr/"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive"
}





In [0]:
import boto3
import requests
import re
import os
from dateutil.parser import parse
from pyspark.sql import functions as F


In [0]:


def list_source_files():
    url = BASE_URL

    html = requests.get(url, headers=HEADERS).text
    #print (html)
    #print ('----')
    # Extract ALL href values
    hrefs = re.findall(r'HREF="([^"]+)"', html, flags=re.IGNORECASE)

    # Keep only the ones inside this directory
    return [
        h.split("/")[-1]
        for h in hrefs
        if "/pub/time.series/pr/" in h and not h.endswith("/")
    ]

In [0]:
print (list_source_files())

<html><head><title>download.bls.gov - /pub/time.series/pr/</title></head><body><H1>download.bls.gov - /pub/time.series/pr/</H1><hr>

<pre><A HREF="/pub/time.series/">[To Parent Directory]</A><br><br> 1/29/2026  8:30 AM          102 <A HREF="/pub/time.series/pr/pr.class">pr.class</A><br> 9/13/2022  3:52 PM          562 <A HREF="/pub/time.series/pr/pr.contacts">pr.contacts</A><br> 1/29/2026  8:30 AM      1576038 <A HREF="/pub/time.series/pr/pr.data.0.Current">pr.data.0.Current</A><br> 1/29/2026  8:30 AM      3199632 <A HREF="/pub/time.series/pr/pr.data.1.AllData">pr.data.1.AllData</A><br> 1/29/2026  8:30 AM          176 <A HREF="/pub/time.series/pr/pr.duration">pr.duration</A><br> 1/29/2026  8:30 AM           40 <A HREF="/pub/time.series/pr/pr.footnote">pr.footnote</A><br> 1/29/2026  8:30 AM          745 <A HREF="/pub/time.series/pr/pr.measure">pr.measure</A><br>  1/7/1994  2:53 PM          146 <A HREF="/pub/time.series/pr/pr.period">pr.period</A><br>11/18/2011  3:05 PM           79 <A H

In [0]:


# Initialize S3 client using your variable names
s3 = boto3.client(
    "s3",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name='us-east-1'
)

def sync_to_s3(file_list):
    for filename in file_list:
        file_url = f"{BASE_URL}{filename}"
        s3_key = f"time_series/pr/{filename}"
        
        # 1. Get Source Metadata (HEAD request is fast and saves bandwidth)
        source_head = requests.head(file_url, headers=HEADERS)
        source_size = int(source_head.headers.get('Content-Length', 0))
        source_last_mod = parse(source_head.headers.get('Last-Modified'))

        # 2. Check S3 Metadata
        try:
            s3_meta = s3.head_object(Bucket=BUCKET, Key=s3_key)
            s3_size = s3_meta['ContentLength']
            s3_last_mod = s3_meta['LastModified']
            
            # 3. Compare: Skip if size matches and S3 is NOT older than source
            if source_size == s3_size and s3_last_mod >= source_last_mod:
                print(f"Skipping {filename}: Already up to date.")
                continue
            else:
                print(f"Updating {filename}: Changes detected...")
                
        except s3.exceptions.ClientError:
            # File doesn't exist in S3 yet
            print(f"Uploading {filename}: New file...")

        # 4. Perform the actual upload
        file_res = requests.get(file_url, headers=HEADERS)
        if file_res.status_code == 200:
            s3.put_object(
                Bucket=BUCKET,
                Key=s3_key,
                Body=file_res.content
            )
            print(f"Successfully synced {filename}")



In [0]:
files_to_sync = list_source_files()
sync_to_s3(files_to_sync)


Skipping pr.class: Already up to date.
Skipping pr.contacts: Already up to date.
Skipping pr.data.0.Current: Already up to date.
Skipping pr.data.1.AllData: Already up to date.
Skipping pr.duration: Already up to date.
Skipping pr.footnote: Already up to date.
Skipping pr.measure: Already up to date.
Skipping pr.period: Already up to date.
Skipping pr.seasonal: Already up to date.
Skipping pr.sector: Already up to date.
Skipping pr.series: Already up to date.
Skipping pr.txt: Already up to date.


In [0]:
def cleanup_s3(source_file_list):
    print("Starting cleanup check...")
    
    # 1. List all objects currently in your S3 'folder'
    prefix = "time_series/pr/"
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    
    if 'Contents' not in response:
        print("S3 bucket is empty. Nothing to clean.")
        return

    # 2. Extract the filenames from S3 keys
    # Example: 'time_series/pr/pr.txt' -> 'pr.txt'
    s3_files = {obj['Key'].replace(prefix, "") for obj in response['Contents']}
    
    # 3. Identify files in S3 that are NOT in the source list
    source_files_set = set(source_file_list)
    files_to_delete = s3_files - source_files_set
    
    if not files_to_delete:
        print("No orphaned files found. Bucket is in sync.")
    else:
        for filename in files_to_delete:
            delete_key = f"{prefix}{filename}"
            print(f"Deleting {filename} from S3 (no longer exists at source)...")
            s3.delete_object(Bucket=BUCKET, Key=delete_key)
            print(f"Successfully deleted {delete_key}")

# Run the cleanup
cleanup_s3(files_to_sync)

Starting cleanup check...
No orphaned files found. Bucket is in sync.


In [0]:
def sync_population_data():
    # This is the specific Honolulu-API endpoint found on the GitHub page
    pop_api_url = "https://honolulu-api.datausa.io/tesseract/data.jsonrecords?cube=acs_yg_total_population_1&drilldowns=Year%2CNation&locale=en&measures=Population".strip()
    s3_key = "population_data/us_population.json"
    
    print(f"Fetching population data from DataUSA...")
    
    # Using your HEADERS is still recommended for this domain
    response = requests.get(pop_api_url, headers=HEADERS)
    
    if response.status_code == 200:
        s3.put_object(
            Bucket=BUCKET,
            Key=s3_key,
            Body=response.text 
        )
        print(f"Successfully saved population data to s3://{BUCKET}/{s3_key}")
    else:
        print(f"Failed! Status: {response.status_code}")
        print(f"Message: {response.text[:100]}")

# Execute
sync_population_data()

Fetching population data from DataUSA...
Successfully saved population data to s3://foruhar-rearc-quest/population_data/us_population.json


In [0]:

s3_options = {
    "header": "true",
    "inferSchema": "true",
    "fs.s3a.access.key": access_key,
    "fs.s3a.secret.key": secret_key,
    "fs.s3a.endpoint": "s3.amazonaws.com",
    "ignoreLeadingWhiteSpace": "true",
    "ignoreTrailingWhiteSpace": "true"
}

# 1. Reload BLS Data with the new whitespace options
df_bls = (spark.read
    .format("csv")
    .options(**s3_options)
    .option("sep", "\t")
    .load(f"s3a://{BUCKET}/time_series/pr/pr.data.0.Current"))

# 2. Reload Population Data (just to keep options consistent)
df_pop_raw = (spark.read
    .format("json")
    .options(**s3_options)
    .load(f"s3a://{BUCKET}/population_data/us_population.json"))

# 3. Flatten the population data as before
if "data" in df_pop_raw.columns:
    df_pop = df_pop_raw.select(F.explode("data").alias("record")).select("record.*")
else:
    df_pop = df_pop_raw

# Verify the column names are now clean
print("Cleaned Columns:", df_bls.columns)
df_bls.show(5)

# Quick check
df_bls.show(5)
df_pop.show(5)

Cleaned Columns: ['series_id', 'year', 'period', 'value', 'footnote_codes']
+-----------+----+------+-----+--------------+
|  series_id|year|period|value|footnote_codes|
+-----------+----+------+-----+--------------+
|PRS30006011|1995|   Q01|  2.6|          NULL|
|PRS30006011|1995|   Q02|  2.1|          NULL|
|PRS30006011|1995|   Q03|  0.9|          NULL|
|PRS30006011|1995|   Q04|  0.1|          NULL|
|PRS30006011|1995|   Q05|  1.4|          NULL|
+-----------+----+------+-----+--------------+
only showing top 5 rows
+-----------+----+------+-----+--------------+
|  series_id|year|period|value|footnote_codes|
+-----------+----+------+-----+--------------+
|PRS30006011|1995|   Q01|  2.6|          NULL|
|PRS30006011|1995|   Q02|  2.1|          NULL|
|PRS30006011|1995|   Q03|  0.9|          NULL|
|PRS30006011|1995|   Q04|  0.1|          NULL|
|PRS30006011|1995|   Q05|  1.4|          NULL|
+-----------+----+------+-----+--------------+
only showing top 5 rows
+-------------+---------+-----

In [0]:

# Clean Population Data
df_pop_clean = df_pop.withColumn("Year", F.col("Year").cast("int")) \
                     .withColumn("Population", F.col("Population").cast("long"))

# Filter for 2013-2018 inclusive
df_pop_filtered = df_pop_clean.filter((F.col("Year") >= 2013) & (F.col("Year") <= 2018))

# Calculate Stats
stats = df_pop_filtered.select(
    F.mean("Population").alias("Mean_Population"),
    F.stddev("Population").alias("StdDev_Population")
)
stats.show()

+---------------+-----------------+
|Mean_Population|StdDev_Population|
+---------------+-----------------+
|   3.22069808E8|4158441.040908095|
+---------------+-----------------+



In [0]:
# Clean BLS: Trim the whitespace from series_id and cast value
df_bls_clean = df_bls.withColumn("series_id", F.trim(F.col("series_id"))) \
                     .withColumn("value", F.col("value").cast("double"))

# Sum values per series and year
df_yearly_sum = df_bls_clean.groupBy("series_id", "year").agg(F.sum("value").alias("total_value"))

# Use a Window function to find the max year for each series
from pyspark.sql.window import Window
window_spec = Window.partitionBy("series_id").orderBy(F.desc("total_value"))

df_best_years = df_yearly_sum.withColumn("rank", F.rank().over(window_spec)) \
                             .filter(F.col("rank") == 1) \
                             .select("series_id", "year", "total_value")

df_best_years.show(5)

+-----------+----+------------------+
|  series_id|year|       total_value|
+-----------+----+------------------+
|PRS30006011|2022|              20.5|
|PRS30006012|2022|              17.1|
|PRS30006013|1998|           705.895|
|PRS30006021|2010|              17.7|
|PRS30006022|2010|12.399999999999999|
+-----------+----+------------------+
only showing top 5 rows


In [0]:
# 1. Filter BLS for the target series and period
df_target_series = df_bls_clean.filter(
    (F.col("series_id") == "PRS30006032") & 
    (F.col("period") == "Q01")
)

# 2. Join with the Population data (using the cleaned df from the previous step)
# We join on the 'year' column
df_final_report = df_target_series.join(
    df_pop_clean, 
    df_target_series.year == df_pop_clean.Year, 
    "inner"
).select(
    df_target_series.series_id,
    df_target_series.year,
    df_target_series.period,
    df_target_series.value,
    df_pop_clean.Population
).orderBy("year")

# 3. Show the final result
df_final_report.show()

+-----------+----+------+-----+----------+
|  series_id|year|period|value|Population|
+-----------+----+------+-----+----------+
|PRS30006032|2013|   Q01|  0.5| 316128839|
|PRS30006032|2014|   Q01| -0.1| 318857056|
|PRS30006032|2015|   Q01| -1.7| 321418821|
|PRS30006032|2016|   Q01| -1.4| 323127515|
|PRS30006032|2017|   Q01|  0.9| 325719178|
|PRS30006032|2018|   Q01|  0.5| 327167439|
|PRS30006032|2019|   Q01| -1.6| 328239523|
|PRS30006032|2021|   Q01|  0.7| 331893745|
|PRS30006032|2022|   Q01|  5.3| 333287562|
|PRS30006032|2023|   Q01|  0.3| 334914896|
+-----------+----+------+-----+----------+

